# Script to convert netCDF climate files from the Met Office Unified Model into PSG GCM files compatible with the GlobES tool. Created by Fauchez, Villanueva - NASA Goddard Space Flight Center, March 2021. Scripts amended to also include 3-D Chemistry variables and eccentric orbit (M. Braam, CSH, University of Bern), Oct 2024

In [1]:
# ---------------------------------------------------------------
# Script to convert netCDF climate files into PSG GCM files
# netCDF files in UM general circulation model
# Fauchez, Villanueva - NASA Goddard Space Flight Center
# March 2021
# Amended to also include 3-D Chemistry variables (MB)
# ---------------------------------------------------------------
import sys, os
import numpy as np
from netCDF4 import Dataset as ncdf
import getpass

# Module that performs the conversion
def convertgcm(filein = 'data/gcm_um.nc', fileout = 'PSG/GlobES/spectra/spectra0224/gcm_psg_pcb_11_i70_12_w.dat', itime=12, phase=0.0, sol_lon=0.0):
	# Read the netCDF file
    nfile = ncdf(filein)
#	print(nfile.variables)
    lat = (nfile.variables['latitude'])[:];
    lon = (nfile.variables['longitude'])[:];
    alt = (nfile.variables['hybrid_ht'])[:];
    tsurf = (nfile.variables['temp'])[:]; tsurf = tsurf[itime,:,:]
    temp = (nfile.variables['temp_1'])[:];  temp = temp[itime,:,:,:]
    press = (nfile.variables['p'])[:]; press = press[itime,:,:,:]
    uk = (nfile.variables['u'])[:];    uk = uk[itime,:,:,:]
    vk = (nfile.variables['v'])[:];    vk = vk[itime,:,:,:]
    h2o = (nfile.variables['q'])[:];   h2o = h2o[itime,:,:,:] * 28./18.
    ice = (nfile.variables['QCF'])[:];   ice = ice[itime,:,:,:]
    liq = (nfile.variables['QCL'])[:];   liq = liq[itime,:,:,:]
    o3 = (nfile.variables['field2101'])[:];   o3 = o3[itime,:,:,:] *28./48
    no = (nfile.variables['field2102'])[:];   no = no[itime,:,:,:] *28./30.01
    no2 = (nfile.variables['field3102'])[:];   no2 = no2[itime,:,:,:] *28./46
    n2o = (nfile.variables['field2149'])[:];   n2o = n2o[itime,:,:,:] *28./44
    hno3 = (nfile.variables['field2107'])[:];   hno3 = hno3[itime,:,:,:] *28./63
    print(nfile.variables['QCF'])
   
#	rliq = (nfile.variables['STASH_m01s01i221'])[:];  rliq = rliq[itime,:,:,:]/1e6# Size of ice clouds in m
    nfile.close()
	# Fix variables
    sz = np.shape(temp)
    print(sz)
    p = np.zeros(sz); pw = p[:-1,:,:]
    h2ow = np.zeros(sz); h2ow = h2o[:-1,:,:]
    icew = np.zeros(sz); icew = ice[:-1,:,:]
    liqw = np.zeros(sz); liqw = liq[:-1,:,:]
#	uw = np.zeros(sz); uw[:-1,:,:] = uk
    vw = np.zeros(sz); vw[:,:,:] = vk[:,:-1,:]
#	print(vw)
    tsurf = np.where((tsurf>0) & (np.isfinite(tsurf)), tsurf, 300.0)
    temp = np.where((temp>0) & (np.isfinite(temp)), temp, 300.0)
    h2ow = np.where((h2ow>0) & (np.isfinite(h2ow)), h2ow, 1e-30)
    icew = np.where((icew>0) & (np.isfinite(icew)), icew, 1e-30)
    liqw = np.where((liqw>0) & (np.isfinite(liqw)), liqw, 1e-30)
#	rliq = np.where((rliq>0) & (np.isfinite(rliq)), rliq, 1e-6)
    o3 = np.where((o3>0) & (np.isfinite(o3)), o3, 1e-30)
    no = np.where((no>0) & (np.isfinite(no)), no, 1e-30)
    no2 = np.where((no2>0) & (np.isfinite(no2)), no2, 1e-30)
    n2o = np.where((n2o>0) & (np.isfinite(n2o)), n2o, 1e-30)
    hno3 = np.where((hno3>0) & (np.isfinite(hno3)), hno3, 1e-30)
#
    obs_lon = (360-phase+sol_lon) %360
	# Save object parameters
    newf = []
    newf.append('<OBJECT>Exoplanet')
    newf.append('<OBJECT-NAME>Proxima Cen b')
    newf.append('<OBJECT-DATE>2020/04/08 01:32')
    newf.append('<OBJECT-DIAMETER>14320')
    newf.append('<OBJECT-GRAVITY>10.9')
    newf.append('<OBJECT-GRAVITY-UNIT>g')
    newf.append('<OBJECT-STAR-DISTANCE>0.04850')
    newf.append('<OBJECT-STAR-VELOCITY>0.00132')
    newf.append('<OBJECT-SOLAR-LONGITUDE>%s' %sol_lon)
    newf.append('<OBJECT-SOLAR-LATITUDE>0.00')
    newf.append('<OBJECT-SEASON>%s' %phase)
    newf.append('<OBJECT-STAR-TYPE>M')
    newf.append('<OBJECT-STAR-TEMPERATURE>3050')
    newf.append('<OBJECT-STAR-RADIUS>0.14')
    newf.append('<OBJECT-OBS-LONGITUDE>%s' %obs_lon)
    newf.append('<OBJECT-OBS-LATITUDE>20.00')
    newf.append('<OBJECT-OBS-VELOCITY>-21.100')
    newf.append('<OBJECT-PERIOD>11.186')
    newf.append('<OBJECT-STAR-METALLICITY>0.0')
    newf.append('<OBJECT-INCLINATION>70')
    newf.append('<OBJECT-ORBIT>0.0,0.00000,11.186,0.00000,0.04544,2455701.776757')#0.0,0.00000,11.186,0.00000,0.04544,2455701.776757')#0.30000000,0.00000,11.18680000,3.30000,0.03395,2457900.523121
    newf.append('<OBJECT-POSITION-ANGLE>0.00')
    newf.append('<OBJECT-SHAPE>Sphere')
    newf.append('<GEOMETRY>Observatory')
    newf.append('<GEOMETRY-OFFSET-NS>0.0')
    newf.append('<GEOMETRY-OFFSET-EW>0.0')
    newf.append('<GEOMETRY-OFFSET-UNIT>arcsec')
    newf.append('<GEOMETRY-OBS-ALTITUDE>85')
    newf.append('<GEOMETRY-ALTITUDE-UNIT>km')
    newf.append('<GEOMETRY-USER-PARAM>0.0')
    newf.append('<GEOMETRY-STELLAR-TYPE>G')
    newf.append('<GEOMETRY-STELLAR-TEMPERATURE>5777')
    newf.append('<GEOMETRY-STELLAR-MAGNITUDE>0')
    newf.append('<GEOMETRY-SOLAR-ANGLE>81.569')
    newf.append('<GEOMETRY-OBS-ANGLE>48.169')
    newf.append('<GEOMETRY-PLANET-FRACTION>1.000e+00')
    newf.append('<GEOMETRY-STAR-DISTANCE>3.368696e-02')
    newf.append('<GEOMETRY-STAR-FRACTION>0.000000e+00')
    newf.append('<GEOMETRY-REF>User')
    newf.append('<GEOMETRY-DISK-ANGLES>1')
    newf.append('<GEOMETRY-ROTATION>-0.00000,0.0')
    newf.append('<GEOMETRY-AZIMUTH>0.000')
    newf.append('<GEOMETRY-BRDFSCALER>1.000')
    newf.append('<GEOMETRY-TEMISSION>272.296')#272.296

	# Save atmosphere parameters
    newf.append('<ATMOSPHERE-DESCRIPTION>Met Office Unified Model (UM) simulation')
    newf.append('<ATMOSPHERE-STRUCTURE>Equilibrium')
    newf.append('<ATMOSPHERE-PRESSURE>100000')
    newf.append('<ATMOSPHERE-PUNIT>Pa')
    newf.append('<ATMOSPHERE-WEIGHT>28.0')
    newf.append('<ATMOSPHERE-LAYERS>60')
    newf.append('<ATMOSPHERE-NGAS>9')
    newf.append('<ATMOSPHERE-GAS>N2,O2,CO2,H2O,O3,NO,NO2,N2O,HNO3')
    newf.append('<ATMOSPHERE-TYPE>HIT[22],HIT[7],HIT[2],HIT[1],HIT[3],HIT[8],HIT[10],HIT[4],HIT[12]')
    newf.append('<ATMOSPHERE-ABUN>78,21,400,1,1,1,1,1,1')
    newf.append('<ATMOSPHERE-UNIT>pct,pct,ppm,scl,scl,scl,scl,scl,scl')
    newf.append('<ATMOSPHERE-NMAX>2')
    newf.append('<ATMOSPHERE-LMAX>2')
    newf.append('<ATMOSPHERE-NAERO>2')
    newf.append('<ATMOSPHERE-AEROS>Water,WaterIce')
    newf.append('<ATMOSPHERE-ATYPE>AFCRL_Water_HRI,Warren_ice_HRI')
    newf.append('<ATMOSPHERE-AABUN>1,1')
    newf.append('<ATMOSPHERE-AUNIT>scl,scl')
    newf.append('<ATMOSPHERE-ASIZE>5,100')
    newf.append('<ATMOSPHERE-ASUNI>um,um')

	# Save surface parameters
    newf.append('<SURFACE-TEMPERATURE>272.296')#272.296
    newf.append('<SURFACE-ALBEDO>0.2')
    newf.append('<SURFACE-EMISSIVITY>1.0')
    newf.append('<SURFACE-NSURF>0')
    newf.append('<SURFACE-GAS-RATIO>1.0')
    newf.append('<SURFACE-NSURF>0')
    newf.append('<SURFACE-GAS-UNIT>ratio')
    newf.append('<SURFACE-MODEL>Lambert')
    newf.append('<SURFACE-PHASEMODEL>ISO')
    newf.append('<GENERATOR-BEAM>1')
    newf.append('<GENERATOR-BEAM-UNIT>diameter')
    newf.append('<GENERATOR-GAS-MODEL>Y')
    newf.append('<GENERATOR-CONT-MODEL>N')
    newf.append('<GENERATOR-DIAMTELE>6.5')
    newf.append('<GENERATOR-LOGRAD>N')
    newf.append('<GENERATOR-CONT-STELLAR>N')
    newf.append('<GENERATOR-TRANS-APPLY>N')
    newf.append('<GENERATOR-TRANS-SHOW>N')
    newf.append('<GENERATOR-TRANS>02-01')
    newf.append('<GENERATOR-NOISE>CCD')
    newf.append('<GENERATOR-NOISE2>0.05')
    newf.append('<GENERATOR-NOISETIME>0.4')
    newf.append('<GENERATOR-NOISEOTEMP>50')
    newf.append('<GENERATOR-NOISEOEFF>0.2')
    newf.append('<GENERATOR-NOISEOEMIS>0.1')
    newf.append('<GENERATOR-NOISEFRAMES>4000')
    newf.append('<GENERATOR-NOISEPIXELS>8')
    newf.append('<GENERATOR-NOISE1>16.8')
    newf.append('<GENERATOR-TELESCOPE>SINGLE')
    newf.append('<GENERATOR-TELESCOPE1>1')
    newf.append('<GENERATOR-TELESCOPE2>2.0')
    newf.append('<GENERATOR-TELESCOPE3>1.0')
    newf.append('<GENERATOR-INSTRUMENT>user')
    newf.append('<GENERATOR-RESOLUTIONKERNEL>N')
	# Save GCM parameters
    vars = '<ATMOSPHERE-GCM-PARAMETERS>'
    vars = vars + str("%d,%d,%d,%.1f,%.1f,%.2f,%.2f" %(sz[2],sz[1],sz[0],lon[0],lat[0],lon[1]-lon[0],lat[1]-lat[0]))
    vars = vars + ',Winds,Tsurf,Temperature,Pressure,H2O,Water,WaterIce,O3,NO,NO2,N2O,HNO3'
    newf.append(vars)
    with open(fileout,'w') as fw:
        for i in newf: fw.write(i+'\n')
    with open(fileout,'ab') as fb:
        if sys.version_info>=(3,0,0): bc=fb.write(bytes('<BINARY>',encoding = 'utf-8'))
        else: bc=fb.write('<BINARY>')
        fb.write(np.asarray(uk,order='C',dtype=np.float32))
        fb.write(np.asarray(vw,order='C',dtype=np.float32))
        fb.write(np.asarray(tsurf,order='C',dtype=np.float32))
        fb.write(np.asarray(temp,order='C',dtype=np.float32))
        fb.write(np.log10(np.asarray(press*1e-5,order='C',dtype=np.float32)))
        fb.write(np.log10(np.asarray(h2ow,order='C',dtype=np.float32)))
        fb.write(np.log10(np.asarray(liqw,order='C',dtype=np.float32)))
        fb.write(np.log10(np.asarray(icew,order='C',dtype=np.float32)))
        #fb.write(np.log10(np.asarray(rliq,order='C',dtype=np.float32)))
        fb.write(np.log10(np.asarray(o3,order='C',dtype=np.float32)))
        fb.write(np.log10(np.asarray(no,order='C',dtype=np.float32)))
        fb.write(np.log10(np.asarray(no2,order='C',dtype=np.float32)))
        fb.write(np.log10(np.asarray(n2o,order='C',dtype=np.float32)))
        fb.write(np.log10(np.asarray(hno3,order='C',dtype=np.float32)))
        if sys.version_info>=(3,0,0): bc=fb.write(bytes('</BINARY>',encoding = 'utf-8'))
        else: bc=fb.write('</BINARY>')
    fb.close()
#End convert

#if __name__ == "__main__": convertgcm(filein='data/pcb_11_daily_7458_1.nc', fileout = 'PSG/GlobES/spectra/spectra0224/gcm_psg_pcb_11_i70_12_w.dat', itime=12) #pcb_32res_6120_120, pcb_11_daily_7458_1, pcb_nox_7560_120_ste

# Scripts to recalculate orbital evolution of Proxima Centauri b, following the implementation of orbital astronomy in the Unified Model and SOCRATES radiative transfer (Edwards & Slingo, 1996, QJRMS; Manners et al., 2021, SOCRATES technical guide; Braam & Angerhausen, 2025, A&A)

In [3]:
import iris
import numpy as np

def UM_orbits(cubes, t_slice=0,t_0=3.1985745555555556,t_sim=17000,P=11.186,e=0.3):
    max_toa_sw = {}
    # import netcdf cubes to check incoming flux calculation with UM output
    for cube in cubes:
        if cube.standard_name in ['toa_incoming_shortwave_flux']: #, 'surface_temperature'
            # Extract the time slice and copy it
            data_slice = cube[t_slice, 0, :, :].copy()
            
            # Find the maximum value of TOA SW flux and its indices
            max_value = data_slice.data.max()
            max_index = (data_slice.data == max_value).nonzero()

            # Extract the corresponding latitude and longitude
            lat_coord = data_slice.coord('latitude')
            lon_coord = data_slice.coord('longitude')

            # Retrieve the exact grid points from indices
            latitude = lat_coord.points[max_index[0]]
            longitude = lon_coord.points[max_index[1]]

            # Store the coordinates of max TOA SW flux in dictionary
            max_toa_sw[cube.standard_name] = {
                'max_value': max_value,
                'latitude': latitude,
                'longitude': longitude
            }
            
            print(f"Current Time slice:: {t_slice} days")
            # first current time in Julian dates (See UM023 radiation paper and source code)
            # uses Julian dates for 00:00:00 01-01-0001, plus planetary epoch 2451545 (00:30:00 UT, 01-01-2013)
            # t_slice for start of cube contents, t_sim for spin-up time (17000 days for 3:2), t_0 for time to longitude of perihelion
            day_nr_midnight = 1721425.5 + 365*(2000-1) + (2000-1)/4 - (2000-1)/100 + (2000-1)/400 + t_slice +t_sim-t_0-1.0 - 2451545
            # Convert to noon on current day (86400 seconds per day)
            day_number = day_nr_midnight + (43200+240/2)/86400
            #print(day_nr_midnight)
            print(f"The current day number: {day_number} days")
            # Mean anomaly in radians
            m_a_rad =  (6.2400214 +0.56170034*(day_nr_midnight)) % (2*np.pi) # -np.pi (+t_0)
            print(f"Mean anomaly: {np.degrees(m_a_rad)} deg")
            
            # gamph represents pi minus longitude of perihelion (radians)
            gamph = np.pi - 1.796601474

            # hour angle for equation of time later (radians)
            hour_angle = (0.0 + 0.28085*(day_nr_midnight))# % (2 * np.pi)
            
            # True anomaly approximation (radians), from Smart (1944)
            v_a_rad = (m_a_rad +
               (2 * e - (e**3) / 4) * np.sin(m_a_rad) +
               (5 / 4) * (e**2) * np.sin(2 * m_a_rad) +
               (13 / 12) * (e**3) * np.sin(3 * m_a_rad))% (2 * np.pi)
            v_a_deg = np.degrees(v_a_rad % (2*np.pi))
            print(f"True anomaly: {(v_a_deg)} deg") 

            
            S = 2.0739783/(0.0485**2)*(1/(1-e**2))**2*((1+e*np.cos(v_a_rad)))**2 #+e**2/2)
            
            sindec = np.sin(0)*np.sin(v_a_rad-gamph)

            # Calculation of the equation of time for discrepancy between mean solar and clock time
            # following Mueller (Acta Phys. Polo. A 88 Supplement, S-49, 1995)
            epsilon = 0 # Obliquity, 0 here
            y = (np.tan(0.5*epsilon))**2 
            y2 = y*y
            e2 = e**2
            p = 0.5*np.pi - gamph
            eqt = (-y * (1.0-4.0*e2) * np.sin(2*(m_a_rad+p)) - 2.0*e*np.sin(m_a_rad) + 
                   2.0*e*y*np.sin(m_a_rad+2*p) - 2.0*e*y*np.sin(3.0*m_a_rad+2.0*p) -
                   0.5*y2*np.sin(4.0*(m_a_rad+p)) - 1.25*e2*np.sin(2.0*m_a_rad) +
                   2.0*e*y2*np.sin(3.0*m_a_rad+4.0*p) - 3.25*e2*y*np.sin(4.0*m_a_rad+2.0*p)
                   - y2*y*np.sin(6.0*(m_a_rad+p))/3.0)
            # correction for planets hour angle from eq of time
            eqt_f = eqt + (hour_angle) % (2*np.pi) # + np.pi
            print(f"Corrected hour angle: {np.degrees(eqt_f)} deg") 
            #print(f"Hour angle: {(np.degrees(hour_angle)) %360} deg") 
            # Orbital phase angle as seen by distant observer
            phase_angle = (v_a_rad - gamph + np.pi + 0) % (2*np.pi)
            print(f"Phase angle: {np.degrees(phase_angle)} deg") 
            print(f'Calculated S: {S} W/m2')

            print(f"Maximum {cube.standard_name.replace('_', ' ').title()}: {max_value} W/m2")
            print(f"Grid point with maximum value - Latitude: {latitude}, Longitude: {longitude}")
    
    return np.degrees(phase_angle), (np.degrees(eqt_f) % 360), longitude, v_a_deg

# Example usage
# results = max_latlon(cubes, t_slice=0)
# print(results)


# Import relevant NetCDF files for 1:1 and 3:2 SOR, available on request

In [4]:
pcb_11 = iris.load('../../UMData/PCb_LIFe/pcb_11_daily_7458_1.nc')
pcb_32 = iris.load('../../UMData/PCb_LIFe/pcb_32_fix_17120_120.nc')
def newheight(cubes, max_tslice=-1):            
    for cube in cubes:
        if cube.coords()[1].long_name=='Hybrid height':
            height_new = cube.coord('Hybrid height')
            height_new.rename('level_height')
newheight(pcb_11)
newheight(pcb_32)

# Calculate orbital phase angles, hour angles, longitudes of substellar point, and mean anomaly for 1:1 and 3:2 SOR

In [6]:

phase_angles_11 = []
hour_angles_11 = []
longitude_sp_11 = []
v_a_deg_11 = []

for i in range(12):
    phase_angle, eqt_f, longitude, v_a_deg = UM_orbits(pcb_11, t_slice=i, t_sim=7400, e=0.0)
    phase_angles_11.append(float(phase_angle))
    hour_angles_11.append(float(eqt_f))
    longitude_sp_11.append(float(longitude[1]))
    v_a_deg_11.append(float(v_a_deg))
    #phase_angles_32test.append(UM_orbits(pcb_32, t_slice=i, t_sim=17000)[0])
    #phase_angles_32test.append(UM_orbits(pcb_32, t_slice=i, t_sim=17000)[1])

phase_angles_32 = []
hour_angles_32 = []
longitude_sp_32 = []
v_a_deg_32 = []

for i in range(24):
    phase_angle, eqt_f, longitude, v_a_deg = UM_orbits(pcb_32, t_slice=i, t_sim=17000)
    phase_angles_32.append(float(phase_angle))
    hour_angles_32.append(float(eqt_f))
    v_a_deg_32.append(float(v_a_deg))
    longitude_sp_32.append(360-float(eqt_f))
    #phase_angles_32test.append(UM_orbits(pcb_32, t_slice=i, t_sim=17000)[0])
    #phase_angles_32test.append(UM_orbits(pcb_32, t_slice=i, t_sim=17000)[1])

Current Time slice:: 0 days
The current day number: 7396.560314333149 days
Mean anomaly: 65.32642066742413 deg
True anomaly: 65.32642066742413 deg
Corrected hour angle: 213.82772548460858 deg
Phase angle: 168.26410259460684 deg
Calculated S: 881.6997768094376 W/m2
Maximum Toa Incoming Shortwave Flux: 881.355712890625 W/m2
Grid point with maximum value - Latitude: [-1. -1.  1.  1.], Longitude: [  1.25 358.75   1.25 358.75]
Current Time slice:: 1 days
The current day number: 7397.560314333149 days
Mean anomaly: 97.50947950050144 deg
True anomaly: 97.50947950050144 deg
Corrected hour angle: 229.91924516086348 deg
Phase angle: 200.44716142768414 deg
Calculated S: 881.6997768094376 W/m2
Maximum Toa Incoming Shortwave Flux: 881.355712890625 W/m2
Grid point with maximum value - Latitude: [-1. -1.  1.  1.], Longitude: [  1.25 358.75   1.25 358.75]
Current Time slice:: 2 days
The current day number: 7398.560314333149 days
Mean anomaly: 129.69253833357874 deg
True anomaly: 129.69253833357874 deg

In [7]:
phase_angles_32

[261.57083479253066,
 280.1459974632812,
 298.09951977815,
 318.5681974160531,
 341.73200715322105,
 11.977454409802794,
 58.3668481829887,
 118.0750577782967,
 172.81119084178604,
 210.38290593297955,
 236.02068113703558,
 257.8035095044278,
 276.85959782558297,
 294.61703989655103,
 314.55544690227435,
 337.1424075018649,
 5.353445663447506,
 48.36451577864117,
 106.72261799570934,
 163.8243008771155,
 204.61611681807727,
 231.68389122020744,
 253.93658869753074,
 273.51807257281166]

In [8]:
for i in range(24):
    print(f'{longitude_sp_32[i]:.2f}')
#for i in range(24):
#    print((360-hour_angles_32[i]))

122.21
93.42
64.98
35.43
8.53
352.55
352.11
1.48
7.37
359.24
336.82
307.72
278.67
250.36
220.94
192.96
174.36
171.13
179.57
187.14
181.98
161.77
133.25
103.96


In [9]:
for i in range(24):
    print(f'{v_a_deg_32[i]:.2f}')

#print(f"Corrected hour angle: {np.degrees(eqt_f):.2f} deg")

158.63
177.21
195.16
215.63
238.79
269.04
315.43
15.14
69.87
107.45
133.08
154.87
173.92
191.68
211.62
234.20
262.42
305.43
3.78
60.89
101.68
128.75
151.00
170.58


# Convert the UM output into GlobES files with the correct orbital parameters


In [106]:
# sol_lon_list=[0,16.09,32.18,48.27,64.36,80.45,96.54,112.63,128.72,144.81,160.90,176.99,193.08]
#for i in range(0,13):
#   convertgcm(filein='../../UMData/PCb_LIFe/pcb_11_daily_7458_1.nc', fileout = 'GlobES_files/gcm_psg_pcb_11_i70_%s_phase_%s.dat' %(i+7400, int(phase_angles_11[i])), itime=i, phase=phase_angles_11[i]) #pcb_32res_6120_120
for i in range(0,24):
    convertgcm(filein='../../UMData/PCb_LIFe/pcb_32_fix_17120_120.nc', fileout = 'GlobES_files/gcm_psg_pcb_32_i70_%s_w.dat' %(i+17000), itime=i, phase=phase_angles_32[i],sol_lon=longitude_sp_32[i]) #pcb_32res_6120_120, pcb_11_daily_7458_1

<class 'netCDF4._netCDF4.Variable'>
float32 QCF(t, hybrid_ht_2, latitude, longitude)
    source: Unified Model Output (Vn12.0):
    name: QCF
    title: QCF AFTER TIMESTEP
    date: 25/09/24
    time: 00:00
    long_name: QCF AFTER TIMESTEP
    standard_name: mass_fraction_of_cloud_ice_in_air
    units: kg kg-1
    missing_value: 2e+20
    _FillValue: 2e+20
    valid_min: 0.0
    valid_max: 0.00077956624
unlimited dimensions: t
current shape = (120, 61, 90, 144)
filling on
(60, 90, 144)
<class 'netCDF4._netCDF4.Variable'>
float32 QCF(t, hybrid_ht_2, latitude, longitude)
    source: Unified Model Output (Vn12.0):
    name: QCF
    title: QCF AFTER TIMESTEP
    date: 25/09/24
    time: 00:00
    long_name: QCF AFTER TIMESTEP
    standard_name: mass_fraction_of_cloud_ice_in_air
    units: kg kg-1
    missing_value: 2e+20
    _FillValue: 2e+20
    valid_min: 0.0
    valid_max: 0.00077956624
unlimited dimensions: t
current shape = (120, 61, 90, 144)
filling on
(60, 90, 144)
<class 'netCDF4.